# 01 — Fit the Lennard-Jones scale to Argon

This notebook makes the two rescalings explicit:

1. **Temperature/energy scale:** fit `epsilon/kB` from the dimensionless coexistence
   ratio $\rho_\mathrm{liquid}/\rho_\mathrm{vapor}$.  The length scale cancels.
2. **Density/length scale:** at the mapped temperatures, fit a common density
   multiplier.  Because density scales as $1/\sigma^3$, this determines `sigma`.

Every stage shows raw tables, raw plots, fitted overlays, and residuals.  The fitted
scale and exact fit points are saved for later notebooks.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from md_Helpers import (
    ProjectPaths,
    SQLiteRunDatabase,
    fit_argon_lj_scale,
    query_argon_scaling_states,
)
from md_Helpers.argon_scaling import default_argon_saturation_path

MD_REPO = Path.home() / "MDsims"
ARGON_DIRECTORY = MD_REPO / "Argon"
DATA_DIRECTORY = ARGON_DIRECTORY / "data"
PLOT_DIRECTORY = ARGON_DIRECTORY / "plots"
DATA_DIRECTORY.mkdir(parents=True, exist_ok=True)
PLOT_DIRECTORY.mkdir(parents=True, exist_ok=True)

database = SQLiteRunDatabase(ProjectPaths().database)
print("Database:", database.path)


## Raw MD and Argon coexistence data


In [ ]:
states = query_argon_scaling_states(database).sort_values(
    ["Therm_kT", "Run_ID"]
).reset_index(drop=True)

argon_path = default_argon_saturation_path()
argon = pd.read_csv(argon_path)
argon["density_ratio"] = (
    argon["density_liquid_mol_L"] / argon["density_vapor_mol_L"]
)

print("Selected MD states:", len(states))
print("MD temperatures:", sorted(states["Therm_kT"].unique()))
print("Argon table:", argon_path)
print("Argon temperature range:", argon["temperature_K"].min(), "to", argon["temperature_K"].max(), "K")

states[[
    "Run_ID", "Therm_kT", "rho_liquid", "rho_liquid_unc",
    "rho_gas", "rho_gas_unc",
]]


In [ ]:
argon[[
    "temperature_K", "density_liquid_mol_L",
    "density_vapor_mol_L", "density_ratio",
]]


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

axes[0].errorbar(states["Therm_kT"], states["rho_liquid"], yerr=states["rho_liquid_unc"],
                 fmt="o", capsize=2, alpha=0.65, label="MD liquid")
axes[0].errorbar(states["Therm_kT"], states["rho_gas"], yerr=states["rho_gas_unc"],
                 fmt="s", capsize=2, alpha=0.65, label="MD vapor")
axes[0].set(xlabel="Reduced temperature, kT", ylabel="Reduced density", title="Raw MD coexistence data")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(argon["temperature_K"], argon["density_liquid_mol_L"], label="Argon liquid")
axes[1].plot(argon["temperature_K"], argon["density_vapor_mol_L"], label="Argon vapor")
axes[1].set(xlabel="Temperature (K)", ylabel="Density (mol/L)", title="Raw NIST Argon coexistence data", yscale="log")
axes[1].legend()
axes[1].grid(alpha=0.3, which="both")

plt.show()


## Stage 1 — Temperature scale from the density ratio


In [ ]:
ratio_raw = states[[
    "Run_ID", "Therm_kT", "rho_liquid", "rho_liquid_unc",
    "rho_gas", "rho_gas_unc",
]].copy()
ratio_raw["density_ratio"] = ratio_raw["rho_liquid"] / ratio_raw["rho_gas"]
ratio_raw["density_ratio_unc"] = ratio_raw["density_ratio"] * np.sqrt(
    (ratio_raw["rho_liquid_unc"] / ratio_raw["rho_liquid"]) ** 2
    + (ratio_raw["rho_gas_unc"] / ratio_raw["rho_gas"]) ** 2
)
ratio_raw


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
axes[0].errorbar(ratio_raw["Therm_kT"], ratio_raw["density_ratio"],
                 yerr=ratio_raw["density_ratio_unc"], fmt="o", capsize=2, alpha=0.7)
axes[0].set(xlabel="Reduced MD temperature, kT", ylabel="Liquid/vapor density ratio",
            title="Raw MD ratio")
axes[0].grid(alpha=0.3)

axes[1].plot(argon["temperature_K"], argon["density_ratio"], color="black")
axes[1].set(xlabel="Temperature (K)", ylabel="Liquid/vapor density ratio",
            title="Raw Argon ratio")
axes[1].grid(alpha=0.3)
plt.show()


In [ ]:
fit = fit_argon_lj_scale(states)
scale = fit.scale

temperature_fit_table = fit.ratio_data[[
    "Run_ID", "Therm_kT", "temperature_K", "rho_liquid_to_gas",
    "ratio_unc", "argon_ratio", "ratio_weighted_residual",
]].rename(columns={
    "rho_liquid_to_gas": "MD_density_ratio",
    "ratio_unc": "MD_density_ratio_unc",
    "argon_ratio": "Argon_density_ratio",
})

print(f"epsilon/kB = {scale.epsilon_over_kb_K:.6f} +/- {scale.epsilon_over_kb_uncertainty_K:.6f} K")
print(f"temperature reduced chi2 = {fit.temperature_reduced_chi2:.3f}")
temperature_fit_table


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)

axes[0].plot(argon["temperature_K"], argon["density_ratio"], color="black", linewidth=2,
             label="NIST Argon")
axes[0].errorbar(temperature_fit_table["temperature_K"], temperature_fit_table["MD_density_ratio"],
                 yerr=temperature_fit_table["MD_density_ratio_unc"], fmt="o", capsize=2,
                 alpha=0.7, label="Mapped MD")
axes[0].set(xlabel="Physical temperature (K)", ylabel="Liquid/vapor density ratio",
            title=f"Temperature fit: epsilon/kB = {scale.epsilon_over_kb_K:.4f} K")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].axhline(0, color="black", linewidth=1)
axes[1].axhline(2, color="tab:red", linestyle="--", alpha=0.6)
axes[1].axhline(-2, color="tab:red", linestyle="--", alpha=0.6)
axes[1].scatter(temperature_fit_table["temperature_K"],
                temperature_fit_table["ratio_weighted_residual"], alpha=0.8)
axes[1].set(xlabel="Mapped physical temperature (K)", ylabel="Weighted residual",
            title=f"Ratio residuals; reduced chi2 = {fit.temperature_reduced_chi2:.2f}")
axes[1].grid(alpha=0.3)
plt.show()


## Stage 2 — Density scale and Lennard-Jones length


In [ ]:
retained = fit.ratio_data.copy()
liquid = pd.DataFrame({
    "Run_ID": retained["Run_ID"], "Therm_kT": retained["Therm_kT"],
    "temperature_K": retained["temperature_K"], "phase": "Liquid",
    "rho_MD": retained["rho_liquid"], "rho_MD_unc": retained["rho_liquid_unc"],
    "rho_Argon_mol_L": retained["argon_liquid_mol_L"],
})
vapor = pd.DataFrame({
    "Run_ID": retained["Run_ID"], "Therm_kT": retained["Therm_kT"],
    "temperature_K": retained["temperature_K"], "phase": "Vapor",
    "rho_MD": retained["rho_gas"], "rho_MD_unc": retained["rho_gas_unc"],
    "rho_Argon_mol_L": retained["argon_vapor_mol_L"],
})
density_fit_table = pd.concat([liquid, vapor], ignore_index=True)
density_fit_table["rho_MD_scaled_mol_L"] = scale.number_density(
    density_fit_table["rho_MD"].to_numpy(dtype=float), "mol/L"
)
density_fit_table["rho_MD_scaled_unc_mol_L"] = scale.number_density_uncertainty(
    density_fit_table["rho_MD"].to_numpy(dtype=float),
    density_fit_table["rho_MD_unc"].to_numpy(dtype=float), "mol/L"
)
density_fit_table["relative_MD_unc"] = density_fit_table["rho_MD_unc"] / density_fit_table["rho_MD"]
density_fit_table["weighted_log_residual"] = np.log(
    density_fit_table["rho_MD_scaled_mol_L"] / density_fit_table["rho_Argon_mol_L"]
) / density_fit_table["relative_MD_unc"]

print(f"density multiplier = {scale.density_mol_L:.6f} mol/L")
print(f"sigma = {scale.sigma_nm:.9f} +/- {scale.sigma_uncertainty_nm:.9f} nm")
print(f"density reduced chi2 = {fit.density_reduced_chi2:.3f}")
density_fit_table


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5.5), constrained_layout=True)

for phase, group in density_fit_table.groupby("phase"):
    axes[0].errorbar(group["rho_MD"], group["rho_Argon_mol_L"], xerr=group["rho_MD_unc"],
                     fmt="o", capsize=2, alpha=0.7, label=phase)
x_line = np.geomspace(density_fit_table["rho_MD"].min() * 0.8,
                      density_fit_table["rho_MD"].max() * 1.2, 300)
axes[0].plot(x_line, scale.density_mol_L * x_line, color="black", linewidth=2,
             label=f"rho_Ar = {scale.density_mol_L:.3f} rho*")
axes[0].set(xlabel="Reduced MD density", ylabel="Matched Argon density (mol/L)",
            title="Common density-scale fit", xscale="log", yscale="log")
axes[0].legend()
axes[0].grid(alpha=0.3, which="both")

axes[1].plot(argon["temperature_K"], argon["density_liquid_mol_L"], color="tab:blue", label="Argon liquid")
axes[1].plot(argon["temperature_K"], argon["density_vapor_mol_L"], color="tab:orange", label="Argon vapor")
styles = {"Liquid": ("tab:blue", "o"), "Vapor": ("tab:orange", "s")}
for phase, group in density_fit_table.groupby("phase"):
    color, marker = styles[phase]
    axes[1].errorbar(group["temperature_K"], group["rho_MD_scaled_mol_L"],
                     yerr=group["rho_MD_scaled_unc_mol_L"], fmt=marker, color=color,
                     capsize=2, alpha=0.65, label=f"Scaled MD {phase.lower()}")
axes[1].set(xlabel="Physical temperature (K)", ylabel="Density (mol/L)",
            title=f"Physical overlay: sigma = {scale.sigma_nm:.6f} nm", yscale="log")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3, which="both")

axes[2].axhline(0, color="black", linewidth=1)
axes[2].axhline(2, color="tab:red", linestyle="--", alpha=0.6)
axes[2].axhline(-2, color="tab:red", linestyle="--", alpha=0.6)
for phase, group in density_fit_table.groupby("phase"):
    axes[2].scatter(group["temperature_K"], group["weighted_log_residual"], label=phase, alpha=0.75)
axes[2].set(xlabel="Mapped physical temperature (K)", ylabel="Weighted log residual",
            title=f"Density residuals; reduced chi2 = {fit.density_reduced_chi2:.1f}")
axes[2].legend()
axes[2].grid(alpha=0.3)
plt.show()


## Replicate scatter and fit-quality check


In [ ]:
replicate_summary = states.groupby("Therm_kT").agg(
    runs=("Run_ID", "count"),
    liquid_mean=("rho_liquid", "mean"), liquid_std=("rho_liquid", "std"),
    mean_liquid_reported_unc=("rho_liquid_unc", "mean"),
    gas_mean=("rho_gas", "mean"), gas_std=("rho_gas", "std"),
    mean_gas_reported_unc=("rho_gas_unc", "mean"),
).reset_index()
replicate_summary["liquid_scatter_over_reported_unc"] = (
    replicate_summary["liquid_std"] / replicate_summary["mean_liquid_reported_unc"]
)
replicate_summary["gas_scatter_over_reported_unc"] = (
    replicate_summary["gas_std"] / replicate_summary["mean_gas_reported_unc"]
)
replicate_summary


In [ ]:
fig, axis = plt.subplots(figsize=(9, 5), constrained_layout=True)
axis.plot(replicate_summary["Therm_kT"], replicate_summary["liquid_scatter_over_reported_unc"],
          "o-", label="Liquid")
axis.plot(replicate_summary["Therm_kT"], replicate_summary["gas_scatter_over_reported_unc"],
          "s-", label="Vapor")
axis.axhline(1, color="black", linestyle="--", label="Scatter = reported uncertainty")
axis.set(xlabel="Reduced temperature, kT", ylabel="Replicate std / mean reported uncertainty",
         title="Do per-run uncertainties describe run-to-run scatter?")
axis.legend()
axis.grid(alpha=0.3)
plt.show()

if fit.temperature_reduced_chi2 > 2 or fit.density_reduced_chi2 > 2:
    print("WARNING: this is an approximate Argon-like mapping, not a statistically adequate calibration.")


## Save the scale and exact fit inputs


In [ ]:
scale_summary = scale.summary()
scale_record = {
    **{name: float(value) for name, value in scale_summary.items()},
    "temperature_chi2": float(fit.temperature_chi2),
    "temperature_dof": int(fit.temperature_dof),
    "temperature_reduced_chi2": float(fit.temperature_reduced_chi2),
    "density_chi2": float(fit.density_chi2),
    "density_dof": int(fit.density_dof),
    "density_reduced_chi2": float(fit.density_reduced_chi2),
    "number_of_selected_states": int(len(states)),
}

outputs = {
    DATA_DIRECTORY / "argon_lj_scale.json": None,
    DATA_DIRECTORY / "argon_scaling_input_states.csv": states,
    DATA_DIRECTORY / "argon_temperature_fit_points.csv": temperature_fit_table,
    DATA_DIRECTORY / "argon_density_fit_points.csv": density_fit_table,
    DATA_DIRECTORY / "argon_replicate_summary.csv": replicate_summary,
}
(DATA_DIRECTORY / "argon_lj_scale.json").write_text(json.dumps(scale_record, indent=2) + "\n")
for path, table in outputs.items():
    if table is not None:
        table.to_csv(path, index=False)
    print("Saved:", path)
